In [20]:
from looptrace import ImageHandler
import dask.array as da
import numpy as np
import pandas as pd
from looptrace import image_processing_functions as ip
from skimage.registration import phase_cross_correlation
import napari
import os
from skimage.measure import regionprops_table
from skimage.morphology import label, remove_small_objects
from numba import njit
from scipy import ndimage as ndi

In [2]:
def int_var(mask, intensity):
    return np.var(intensity[mask>0])
def histogram(mask, intensity):
    return np.histogram(intensity[mask], bins=4)[0]

In [4]:
H = ImageHandler(r"P:\MitoTrace\2021-08-19_MT001_NikonTiE2_HKWT_2E_MitoTrace\analysis\2021-08-19_MT001_2E_config.yaml")
P, T, C, Z, Y, X = H.images.shape

Images loaded from zarr store:  dask.array<stack, shape=(12, 28, 2, 64, 2304, 2304), dtype=float64, chunksize=(1, 1, 1, 1, 2304, 2304), chunktype=numpy.ndarray>
Images cropped to  (12, 28, 2, 64, 2304, 2304)


In [3]:
import nd2reader
img = nd2reader.ND2Reader(r"P:\MitoTrace\2021-08-19_MT001_NikonTiE2_HKWT_2E_MitoTrace\21-08-19_MT001_Ibidi1E_stack002.nd2")
img.metadata['fields_of_view']
nuc_images = [img.get_frame_2D(v=i, c=1, t=0, z=20) for i in img.metadata['fields_of_view']]

In [8]:
ref_frame = H.config['bead_reference_frame']
ref_ch = H.config['bead_ch']
ds = H.config['course_drift_downsample']
z = 20
drift_res =[]
for p in range(1):
    t_img = H.images[p, ref_frame, ref_ch, z, ::ds, ::ds]
    for t in range(T):
        o_img = H.images[p, t, ref_ch, z, ::ds, ::ds]
        course_drift = phase_cross_correlation(np.array(t_img), np.array(o_img), return_error=False) * ds
        drift_res.append([p, t, *course_drift])
drift_res = pd.DataFrame(drift_res, columns=['position', 'frame', 'y_course', 'x_course'])
        

In [4]:
#nuc_ch = H.config['nuc_channel']
#nuc_frame = H.config['nuc_ref_frame']
#nuc_images = [np.array(H.images[p,nuc_frame,nuc_ch,z,:,:]) for p in range(P)]
nuc_masks = ip.nuc_segmentation(nuc_images, model='nuclei')

creating new log file
2021-11-02 06:46:13,079 [INFO] WRITING LOG OUTPUT TO C:\Users\ellenberg\.cellpose\run.log
2021-11-02 06:46:13,107 [INFO] >>>> using CPU
2021-11-02 06:46:13,180 [INFO] ~~~ FINDING MASKS ~~~
2021-11-02 06:46:34,804 [INFO] 100%|##########| 12/12 [00:21<00:00,  1.80s/it]
2021-11-02 06:46:34,805 [INFO] >>>> TOTAL TIME 21.62 sec


<ipython-input-84-780a73194434>:2: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray
  mito_nucs, _ = ndi.label((nuc_images[0]*(nuc_masks[0]==0))>nuc_int) + np.max(nuc_masks[0])


(2304, 2304)

In [24]:
#nuc_images = np.stack(nuc_images)
#nuc_masks = np.stack(nuc_masks)

mito_nucs = []
for p in range(nuc_images.shape[0]):
    nuc_int = np.mean(nuc_images[p][nuc_masks[p]>0])
    mito_nuc = (nuc_images[p] * (nuc_masks[p] == 0)) > 1.5*nuc_int
    mito_nuc = remove_small_objects(mito_nuc, min_size=1000 )
    mito_nuc = label(mito_nuc)
    mito_index = np.max(nuc_masks[p])
    mito_nuc[mito_nuc>0] += mito_index
    print(mito_nuc.shape)
    mito_nucs.append(mito_nuc)
mito_nucs = np.stack(mito_nucs)
nuc_masks = nuc_masks + mito_nucs
##nucs = np.stack(, nuc_masks, nuc_class])
#np.save(H.config['output_path']+os.sep+H.config['output_prefix']+'nucs.npy', nucs)

(2304, 2304)
(2304, 2304)
(2304, 2304)
(2304, 2304)
(2304, 2304)
(2304, 2304)
(2304, 2304)
(2304, 2304)
(2304, 2304)
(2304, 2304)
(2304, 2304)
(2304, 2304)


In [25]:
n = napari.view_image(nuc_images)
n.add_labels(nuc_masks)
#n.add_labels(mito_nucs)
#n_masks = n.add_labels(nucs[1].copy())
#n_class = n.add_labels(n_class.data)

2021-11-02 17:49:45,397 [WARNING] QWindowsWindow::setGeometry: Unable to set geometry 1280x1214+0+34 (frame: 1302x1270-11-11) on QWidgetWindow/"_QtMainWindowClassWindow" on "\\.\DISPLAY241". Resulting geometry: 1280x1098+0+34 (frame: 1302x1154-11-11) margins: 11, 45, 11, 11 minimum size: 608x549 MINMAXINFO maxSize=0,0 maxpos=0,0 mintrack=1238,1154 maxtrack=0,0)


<Labels layer 'nuc_masks' at 0x2991a501d60>

In [40]:
n_class.data.shape

(12, 2304, 2304)

In [32]:
if not np.allclose(n_masks.data, nucs[1]):
    print('Labels changed.')
    nucs[1] = n_masks.data
    np.save(H.config['output_path']+os.sep+H.config['output_prefix']+'nucs.npy', nucs)

In [48]:
nuc_props = pd.DataFrame()
for p in range(len(nucs[0])):
    nuc_props_p = pd.DataFrame(regionprops_table(label_image = n_masks.data[p], intensity_image = nucs[0, p], properties=('label','bbox', 'max_intensity'), extra_properties=(int_var,histogram)))
    nuc_class_p = pd.DataFrame(regionprops_table(label_image = n_masks.data[p], intensity_image = n_class.data[p], properties=('label','max_intensity')))
    nuc_props_p['fov'] = p
    nuc_props_p['class'] = nuc_class_p['max_intensity'].astype(int)
    nuc_props = nuc_props.append(nuc_props_p)
    #print(nuc_class_p)
nuc_props = nuc_props.reset_index(drop=True)
features = nuc_props[['class', 'max_intensity', 'int_var', 'histogram-0', 'histogram-1', 'histogram-2', 'histogram-3']].to_numpy()

In [62]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X = features[:,1:]
y = features[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=0)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)


In [73]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

regressor = RandomForestRegressor(n_estimators=20, random_state=2)
regressor.fit(X_train, y_train)
y_pred = np.round(regressor.predict(X_test))

print(accuracy_score(y_test, y_pred))
print(regressor.feature_importances_)


0.8968253968253969
[0.22985741 0.11038311 0.11765717 0.07128078 0.35759467 0.11322686]


In [74]:
import joblib
joblib.dump(regressor, r'M:\ChromatinTeam\Image_analysis_models\rf_class_nucs_if1_mitosis2.joblib')

['M:\\ChromatinTeam\\Image_analysis_models\\rf_class_nucs_if1_mitosis2.joblib']

In [ ]:
import joblib
regressor = joblib.load(r"M:\ChromatinTeam\Image_analysis_models\rf_class_nucs_if1_mitosis2.joblib")
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = sc.fit_transform(features)
y_pred = np.round(regressor.predict(X))
nuc_props['class'] = y_pred.astype(int)
class_map = pd.Series(nuc_props['class'].values,index=nuc_props['label']).to_dict()